# NEPSE-Impact-500: Qwen3-8B Unsloth QLoRA

This notebook evaluates base Qwen3-8B zero-shot and three-shot, then trains a
QLoRA adapter with Unsloth on the same frozen manifest used by XLM-R. It
produces deterministic structured JSON and evaluates relevance, event type,
direction, sector, symbol, and evidence selection.

In [ ]:
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install -q 'datasets>=3.2,<4' 'trl>=0.15,<1'   'matplotlib>=3.9,<4' 'seaborn>=0.13,<1' 'tqdm>=4.66,<5'

## 1. Verify the GPU and load the frozen corpus

In [ ]:
import torch
assert torch.cuda.is_available(), "Use a Colab or Kaggle GPU runtime."
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT = Path(os.getenv("MARKET_GYAN_PROJECT", "/content/marketGyan"))
DATA = PROJECT / "data/processed"
SPLITS = DATA / "splits"
OUTPUTS = PROJECT / "outputs"
sys.path.insert(0, str(PROJECT))

from market_gyan.dataset import (
    chronological_group_split,
    dataset_readiness,
    read_jsonl,
    split_manifest,
    validate_dataset,
    write_jsonl,
)

gold_path = DATA / "nepse-impact-500.jsonl"
rows = read_jsonl(gold_path)
issues = validate_dataset(rows)
gate = dataset_readiness(rows)
print(json.dumps(gate, indent=2, ensure_ascii=False))
assert not issues, issues[:3]
assert gate["ready"], gate["errors"]

SPLITS.mkdir(parents=True, exist_ok=True)
manifest_path = SPLITS / "manifest.json"
if not manifest_path.exists():
    frozen = chronological_group_split(rows)
    for name, values in frozen.items():
        write_jsonl(SPLITS / f"{name}.jsonl", values)
    manifest_path.write_text(
        json.dumps(split_manifest(frozen), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert {item["id"] for item in manifest["assignments"]} == {row["id"] for row in rows}
group_splits = {}
for item in manifest["assignments"]:
    previous = group_splits.setdefault(item["duplicateGroupId"], item["split"])
    assert previous == item["split"], "Near-duplicate group crosses split boundaries"
print(manifest["counts"], manifest["sha256"])

In [ ]:
train_rows = read_jsonl(SPLITS / "train.jsonl")
validation_rows = read_jsonl(SPLITS / "validation.jsonl")
test_rows = read_jsonl(SPLITS / "test.jsonl")
print(len(train_rows), len(validation_rows), len(test_rows))

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def plot_counter(axis, counter, title, color, rotate=False):
    items = sorted(counter.items(), key=lambda item: str(item[0]))
    if not items:
        axis.text(0.5, 0.5, "No records", ha="center", va="center")
        axis.set_xticks([])
    else:
        labels, values = zip(*items)
        bars = axis.bar(list(labels), list(values), color=color)
        axis.bar_label(bars, padding=2, fontsize=8)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.2)
    if rotate:
        axis.tick_params(axis="x", rotation=65)

relevance = Counter(row["gold"]["relevance"] for row in rows)
languages = Counter(row["gold"]["language"] for row in rows)
events = Counter(row["gold"]["eventType"] for row in rows)
directions = Counter(
    row["gold"]["impactDirection"]
    for row in rows if row["gold"]["relevance"] != "not_relevant"
)

OUTPUTS.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_counter(axes[0, 0], relevance, "Relevance", "#2563eb")
plot_counter(axes[0, 1], languages, "Language", "#0f766e")
plot_counter(axes[1, 0], events, "Event type", "#7c3aed", rotate=True)
plot_counter(axes[1, 1], directions, "Relevant-record direction", "#dc2626")
plt.tight_layout()
plt.savefig(OUTPUTS / "nepse_impact_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 2. Load Qwen3 through Unsloth and define the structured prompt

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-8B"
MAX_SEQ_LENGTH = 1024
use_bf16 = torch.cuda.is_bf16_supported()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer.pad_token = tokenizer.eos_token

def prompt_for(row):
    numbered = "\n".join(
        f"[{sentence['id']}] {sentence['text']}"
        for sentence in row["sentences"]
    )
    return (
        "Return one NEPSE-Impact-500 JSON object. First classify relevance. "
        "Use only numbered evidence sentence IDs. Do not predict prices or "
        "give investment advice.\n"
        f"Title: {row['title']}\n{numbered}"
    )

## 3. Base-model zero-shot and three-shot evaluation

In [ ]:
from tqdm.auto import tqdm

def generate_json(row, demonstrations=None):
    messages = []
    for example in demonstrations or []:
        messages += [
            {"role": "user", "content": prompt_for(example)},
            {
                "role": "assistant",
                "content": json.dumps(
                    example["gold"], ensure_ascii=False, sort_keys=True
                ),
            },
        ]
    messages.append({"role": "user", "content": prompt_for(row)})
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=512, do_sample=False,
            temperature=None, top_p=None
        )
    raw = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw": raw}

three_shot_examples = [
    next(row for row in train_rows if row["gold"]["relevance"] == value)
    for value in ("direct", "indirect", "not_relevant")
]
zero_shot = []
for row in tqdm(test_rows, desc="Qwen zero-shot", unit="doc"):
    zero_shot.append({"id": row["id"], "prediction": generate_json(row)})

three_shot = []
for row in tqdm(test_rows, desc="Qwen three-shot", unit="doc"):
    three_shot.append({
        "id": row["id"],
        "prediction": generate_json(row, three_shot_examples),
    })
write_jsonl(OUTPUTS / "qwen_base_zero_shot.jsonl", zero_shot)
write_jsonl(OUTPUTS / "qwen_base_three_shot.jsonl", three_shot)

## 4. Format gold JSON for supervised fine-tuning

In [ ]:
from datasets import Dataset

def chat_messages(row):
    return [
        {"role": "user", "content": prompt_for(row)},
        {
            "role": "assistant",
            "content": json.dumps(row["gold"], ensure_ascii=False, sort_keys=True),
        },
    ]

def format_training_text(row):
    return tokenizer.apply_chat_template(
        chat_messages(row),
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

def make_dataset(values, include_hard_negatives=True):
    selected = values if include_hard_negatives else [
        row for row in values if row["gold"]["relevance"] != "not_relevant"
    ]
    return Dataset.from_list([
        {"text": format_training_text(row)}
        for row in selected
    ])

train_data = make_dataset(train_rows)
validation_data = make_dataset(validation_rows)
print(len(train_data), len(validation_data))
print(train_data[0]["text"][:600])

## 5. Attach Unsloth LoRA and train or resume

In [ ]:
from transformers import set_seed
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
set_seed(42)
output_dir = OUTPUTS / "marketgyan-qwen3-8b-unsloth-qlora"
updates_per_epoch = max(1, (len(train_data) + 15) // 16)
print(
    f"Training Unsloth QLoRA: train={len(train_data)}, "
    f"validation={len(validation_data)}, approx_steps={updates_per_epoch * 3}"
)
arguments = SFTConfig(
    output_dir=str(output_dir),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_ratio=0.05,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",
    report_to=[],
    seed=42,
    disable_tqdm=False,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=arguments,
    train_dataset=train_data,
    eval_dataset=validation_data,
)

# Mask user tokens so the adapter learns only the reviewed JSON response.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
import shutil
if output_dir.exists():
    for checkpoint in output_dir.glob("checkpoint-*"):
        shutil.rmtree(checkpoint, ignore_errors=True)
trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
for checkpoint in output_dir.glob("checkpoint-*"):
    shutil.rmtree(checkpoint, ignore_errors=True)

## 6. Deterministic held-out generation with the adapter

In [ ]:
FastLanguageModel.for_inference(model)
adapter_predictions = []
for row in tqdm(test_rows, desc="Qwen adapter test generation", unit="doc"):
    adapter_predictions.append({"id": row["id"], "prediction": generate_json(row)})
write_jsonl(output_dir / "test_predictions.jsonl", adapter_predictions)

## 7. Score and plot all Qwen conditions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from market_gyan.metrics import benchmark_predictions

benchmarks = {
    "zero_shot": benchmark_predictions(test_rows, zero_shot),
    "three_shot": benchmark_predictions(test_rows, three_shot),
    "unsloth_qlora": benchmark_predictions(test_rows, adapter_predictions),
}
(output_dir / "metrics.json").write_text(
    json.dumps(benchmarks, indent=2), encoding="utf-8"
)

labels = ["direct", "indirect", "not_relevant"]
matrix = [
    [benchmarks["unsloth_qlora"]["relevance"]["confusion"][actual][predicted]
     for predicted in labels]
    for actual in labels
]
quality_names = [
    "JSON validity", "grounding", "sector F1", "symbol F1", "evidence F1"
]
quality = [
    benchmarks["unsloth_qlora"]["structuredOutputValidity"],
    benchmarks["unsloth_qlora"]["evidenceGrounding"],
    benchmarks["unsloth_qlora"]["sectorMicroF1"],
    benchmarks["unsloth_qlora"]["symbolMicroF1"],
    benchmarks["unsloth_qlora"]["evidenceSentenceF1"],
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0],
    xticklabels=labels, yticklabels=labels
)
axes[0].set_title("Relevance confusion matrix")
benchmark_names = list(benchmarks.keys())
bars = axes[1].bar(
    benchmark_names,
    [benchmarks[name]["relevance"]["macroF1"] for name in benchmark_names],
)
axes[1].bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
axes[1].set_ylim(0, 1)
axes[1].set_title("Base versus Unsloth QLoRA relevance macro-F1")
bars = axes[2].barh(quality_names, quality)
axes[2].bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
axes[2].set_xlim(0, 1)
axes[2].set_title("Structured-output quality")
plt.tight_layout()
plt.savefig(output_dir / "test_results.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 8. Required ablations

Run a second training job with `include_hard_negatives=False` in
`make_dataset(...)` and compare relevance macro-F1. The RAG-enabled versus
RAG-disabled ablation is run through the system evaluation harness because
current factual knowledge must remain outside model weights.

In [ ]:
history = trainer.state.log_history
fig, axis = plt.subplots(figsize=(7, 4))
train_steps = [row["step"] for row in history if "loss" in row]
train_loss = [row["loss"] for row in history if "loss" in row]
eval_steps = [row["step"] for row in history if "eval_loss" in row]
eval_loss = [row["eval_loss"] for row in history if "eval_loss" in row]
if train_steps:
    axis.plot(train_steps, train_loss, label="train")
if eval_steps:
    axis.plot(eval_steps, eval_loss, label="validation")
axis.set_title("Qwen3-8B Unsloth QLoRA loss")
axis.legend()
plt.savefig(output_dir / "loss.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 9. Archive final adapter, predictions, metrics, and plots

In [ ]:
import shutil
archive = shutil.make_archive(str(output_dir), "zip", output_dir)
print(archive)
# Colab: from google.colab import files; files.download(archive)